# 02 カテゴリ別人気構造の分析

カテゴリ別の商品数・価格・レビューの構造を分析し、**「どのカテゴリがどんな特性を持つか」**を明らかにする。

さらに**「同じカテゴリ内で自治体によって戦略が異なるか」**を確認し、Week 3の自治体クラスタリングに向けた仮説を形成する。

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

from src.analysis.category import category_summary, category_price_distribution, category_shop_analysis

pd.set_option('display.float_format', '{:,.1f}'.format)
DATA_DIR = Path('../data')

df = pd.read_parquet(DATA_DIR / 'processed/products.parquet')
print(f'データ: {len(df):,} 件 / {df["category"].nunique()} カテゴリ')

データ: 32,244 件 / 14 カテゴリ


## 1. カテゴリ別サマリー

In [2]:
summary = category_summary(df)
summary[['category','件数','構成比','価格_中央値','レビュー数_中央値','評価平均']].round(1)

,category,件数,構成比,価格_中央値,レビュー数_中央値,評価平均
0,魚介類,6069,18.8,"15,500.0",4.0,4.5
1,牛肉,4531,14.1,"15,000.0",6.0,4.5
2,果物,3647,11.3,"13,000.0",8.0,4.4
3,旅行・体験,2404,7.5,"50,000.0",1.0,4.6
4,その他,2358,7.3,"21,000.0",1.0,4.5
5,家電・電気製品,2297,7.1,"53,000.0",0.0,4.4
6,米・穀物,2269,7.0,"14,500.0",6.0,4.7
7,日用品・生活雑貨,2050,6.4,"16,000.0",2.0,4.6
8,鶏肉,2003,6.2,"13,000.0",1.0,4.6
9,酒・飲料,1516,4.7,"15,000.0",5.0,4.7


In [3]:
# カテゴリ構成比（ドーナツチャート）
fig = px.pie(
    summary[summary['category'] != 'その他'],
    names='category', values='件数',
    hole=0.4,
    title='返礼品カテゴリ構成比（「その他」除く）',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_traces(textinfo='label+percent')
fig.show()

## 2. カテゴリ別 価格 × レビュー ポジショニング

In [4]:
# 4象限マップ：横軸=価格中央値、縦軸=レビュー数中央値
s = summary[summary['category'] != 'その他'].copy()

fig = px.scatter(
    s,
    x='価格_中央値', y='レビュー数_中央値',
    size='件数', color='評価平均',
    text='category',
    color_continuous_scale='RdYlGn',
    range_color=[4.0, 5.0],
    title='カテゴリ ポジショニングマップ<br><sub>横軸: 価格中央値 ／ 縦軸: レビュー数中央値 ／ バブルサイズ: 商品数 ／ 色: 評価平均</sub>',
    labels={'価格_中央値': '寄付金額 中央値（円）', 'レビュー数_中央値': 'レビュー数 中央値'},
    height=550
)
fig.update_traces(textposition='top center', marker=dict(sizemin=8))

# 中央値で4象限を区切る補助線
med_price = s['価格_中央値'].median()
med_review = s['レビュー数_中央値'].median()
fig.add_hline(y=med_review, line_dash='dot', line_color='lightgray')
fig.add_vline(x=med_price, line_dash='dot', line_color='lightgray')

# 象限ラベル
fig.add_annotation(x=s['価格_中央値'].min()*0.95, y=s['レビュー数_中央値'].max()*0.95,
    text='低価格×高人気<br>（コスパ王道）', showarrow=False, font=dict(color='green', size=10))
fig.add_annotation(x=s['価格_中央値'].max()*0.8, y=s['レビュー数_中央値'].max()*0.95,
    text='高価格×高人気<br>（高級プレミアム）', showarrow=False, font=dict(color='blue', size=10))
fig.add_annotation(x=s['価格_中央値'].min()*0.95, y=med_review*0.2,
    text='低価格×低人気<br>（ニッチ商品）', showarrow=False, font=dict(color='gray', size=10))
fig.add_annotation(x=s['価格_中央値'].max()*0.8, y=med_review*0.2,
    text='高価格×低人気<br>（高額選択型）', showarrow=False, font=dict(color='orange', size=10))

fig.show()

## 3. カテゴリ別 価格帯分布

In [5]:
price_dist = category_price_distribution(df)
price_dist = price_dist.drop(index='その他', errors='ignore')

fig = px.imshow(
    price_dist,
    text_auto='.0f',
    title='カテゴリ × 価格帯 構成比（%）<br><sub>各行の合計が100%</sub>',
    labels=dict(x='価格帯', y='カテゴリ', color='構成比(%)'),
    color_continuous_scale='Blues',
    aspect='auto',
    height=450
)
fig.show()

## 4. カテゴリ内 自治体別の差（中江さんの着眼点）

同じカテゴリでも自治体によって「価格帯」「レビュー数」が大きく異なるか？  
→ これが Week 3 の自治体戦略分類の根拠になる。

In [6]:
shop_analysis = category_shop_analysis(df)

# 牛肉カテゴリでの自治体間比較（上位20自治体）
FOCUS_CATEGORY = '牛肉'
top_shops = (
    shop_analysis[shop_analysis['category'] == FOCUS_CATEGORY]
    .nlargest(20, 'レビュー数_合計')
)

fig = px.scatter(
    top_shops,
    x='価格_中央値', y='レビュー数_合計',
    size='商品数', color='評価平均',
    text='shop_name',
    color_continuous_scale='RdYlGn',
    range_color=[4.0, 5.0],
    title=f'【{FOCUS_CATEGORY}】カテゴリ内 自治体ポジショニング（レビュー数TOP20）',
    labels={'価格_中央値': '価格中央値（円）', 'レビュー数_合計': 'レビュー数合計', '商品数': '商品数'},
    height=550
)
fig.update_traces(textposition='top center', marker=dict(sizemin=8))
fig.show()

In [7]:
# 複数カテゴリで同様に確認
for cat in ['魚介類', '果物', '旅行・体験']:
    top = shop_analysis[shop_analysis['category'] == cat].nlargest(15, 'レビュー数_合計')
    fig = px.bar(
        top,
        x='shop_name', y='レビュー数_合計',
        color='価格_中央値',
        color_continuous_scale='Oranges',
        title=f'【{cat}】レビュー数TOP15自治体（色: 価格中央値）',
        labels={'shop_name': '自治体', 'レビュー数_合計': 'レビュー数合計'},
        height=400
    )
    fig.update_layout(xaxis_tickangle=-40)
    fig.show()

## 5. 分析まとめ・事業者目線の示唆

### 発見
1. **魚介類・牛肉が最多カテゴリ** — 食品2カテゴリで全体の3割超を占める
2. **「低価格×高人気」ゾーン** — 牛肉・魚介類・米は1〜2万円台でレビュー数が多い（王道コスパ帯）
3. **「高価格×高人気」ゾーン** — 旅行・体験・家電は高単価でも一定のレビューを獲得
4. **カテゴリ内の自治体格差** — 同じ「牛肉」でも、ブランド牛（飛騨牛・宮崎牛など）を持つ自治体はレビュー数が突出

### ECコンサルタント視点での示唆
- **差別化の鍵はブランド×ストーリー**: 同一カテゴリ内でレビュー数が多い自治体は「銘柄」や「産地ブランド」が明確
- **価格帯の分散**: 魚介類・牛肉は価格帯が広く、低価格の訳あり品から高額の厳選品まで多層展開している
- **旅行・体験は差別化余地大**: レビュー数が相対的に少なく、体験価値の訴求が弱い自治体が多い可能性